# 01 · Anlamsal Kümeleme ve 6 Öznitelikli Matrisin Çıkarılması (Aşama 1, çevrimdışı)

**Araştırma sorusu (RQ1).** Bir SFT örneğinin "değerini" tanımlayan ucuz, model-tabanlı ve yorumlanabilir öznitelikler
nelerdir ve bu öznitelikler anlamsal çeşitlilikle nasıl ilişkilidir?

**Hipotez.** Uzunluk, olabilirlik (PPL), talimat izleme zorluğu (IFD), belirsizlik (entropi) ve çeşitlilik öznitelikleri
birbirini yalnızca kısmen yansıtır; dolayısıyla tek bir öznitelikle (ör. yalnızca IFD) seçim yapmak bilgi kaybettirir
ve GP ile bileşik bir skor aramak anlamlıdır.

## 1.1 Yöntem

**Adım 1 – Anlamsal tabakalama.** $(x_i,y_i)$ metni `BAAI/bge-small-en-v1.5` ile $\mathbf{e}_i\in\mathbb{R}^{384}$ vektörüne
gömülür (birim norm). `KMeans(n_clusters=20, random_state=42)` ile $c(i)\in\{1,\dots,20\}$ elde edilir ve
$$d_i = 1-\frac{\mathbf{e}_i\cdot\mu_{c(i)}}{\lVert\mu_{c(i)}\rVert}$$
kosinüs uzaklığı saklanır.

**Adım 2 – Altı öznitelik** (`Qwen/Qwen2.5-0.5B-Base`, `torch.no_grad()`; yalnızca ileri geçiş):

| # | Öznitelik | Tanım |
|---|---|---|
| 1 | $L_x$ | $\lvert x_i\rvert$ (istem simgesi) |
| 2 | $L_y$ | $\lvert y_i\rvert$ (yanıt simgesi, EOS dâhil) |
| 3 | $PPL$ | $\exp\!\big(-\tfrac{1}{\lvert y_i\rvert}\sum_t\log\pi_{base}(y_{i,t}\mid x_i,y_{i,<t})\big)$ — istem simgeleri `-100` ile maskeli |
| 4 | $IFD$ | $\mathcal{L}_{CE}(y_i\mid x_i)\,/\,\mathcal{L}_{CE}(y_i)$ |
| 5 | $H_y$ | $-\tfrac{1}{\lvert y_i\rvert}\sum_t\sum_{v\in\mathcal{V}}\pi_{base}(v\mid\cdot)\log\pi_{base}(v\mid\cdot)$ |
| 6 | $D_{div}$ | $TTR_i\cdot d_i$, $TTR_i=\lvert\text{benzersiz yanıt simgeleri}\rvert/\lvert y_i\rvert$ |

IFD'nin paydası olan koşulsuz kayıp $\mathcal{L}_{CE}(y_i)$, istemsiz (yalnızca `[EOS]`+yanıt) ikinci bir ileri geçiş gerektirir;
böylece öznitelik başına yalnızca **iki ileri geçiş**, hiç gradyan hesabı yoktur. Son olarak tüm öznitelikler
`MinMaxScaler(feature_range=(0,1))` ile normalize edilir.

> **Uygulama notu.** `lm_head` yalnızca yanıt konumlarına uygulanır (≈150 bin sözcüklük çıktıda bellek tasarrufu);
> toplu işler simge bütçesine göre uzunluğa göre sıralanarak oluşturulur. İstem 256, toplam 512 simgede kırpılır
> (`raw_lx`, `raw_ly` kırpma **öncesi** uzunluklardır).

In [ ]:
# --- Ortam kurulumu (Colab + yerel) ---------------------------------------------------
import os, sys, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)

# Colab uzantısında çekirdek uzak makinede çalışır; yerel src/ orada olmadığından depo klonlanır.
if "google.colab" in sys.modules:
    PROJECT = Path("/content/gp-stratified-sft-selection")
    if not PROJECT.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/suleymangzy/gp-stratified-sft-selection.git", str(PROJECT)], check=True)
else:
    PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ["PROJECT_ROOT"] = str(PROJECT)                  # çıktılar PROJECT/outputs altına yazılır
sys.path.insert(0, str(PROJECT / "src"))

%matplotlib inline
import numpy as np, pandas as pd
from IPython.display import display, Markdown
from analysis import set_academic_style, show_figure, show_table
set_academic_style()
print("Proje dizini:", PROJECT)

## 1.2 Yapılandırma ve çalıştırma

In [ ]:
LIMIT = 0        # 0 = tüm havuz (≈51.760). Hızlı deneme için örn. 2000
FORCE = False    # True: diskteki çıktıları yok sayıp yeniden hesapla

from features import run_feature_extraction
pool, scaled = run_feature_extraction(limit=LIMIT, force=FORCE)
print(f"\nHavuz: {len(pool):,} örnek | sütunlar: {list(pool.columns)}")

## 1.3 Doğrulama (sağlık kontrolleri)
Öznitelik tanımlarının matematiksel olarak beklenen özelliklerini sağlayıp sağlamadığı otomatik denetlenir.

In [ ]:
from analysis import feature_sanity
chk = feature_sanity(pool, scaled)
show_table(chk, "tab_feature_sanity", "Öznitelik matrisi için otomatik sağlık kontrolleri.", floatfmt="{}")
assert (chk["sonuç"] == "geçti").all(), "Bir veya daha fazla kontrol başarısız!"

## 1.4 Bulgular

### 1.4.1 Betimsel istatistikler

In [ ]:
from analysis import feature_summary, plot_feature_distributions
show_table(feature_summary(pool), "tab_feature_summary",
           "Ham (ölçeklenmemiş) özniteliklerin betimsel istatistikleri (N = havuz boyutu). 5% ve 95% yüzdelik dilimlerdir.",
           floatfmt="{:,.3f}")
show_figure(plot_feature_distributions(pool), "fig_feature_distributions",
            "Altı özniteliğin dağılımı (uç %0,5'lik kuyruklar kırpılarak gösterilmiştir; kesikli çizgi medyan).")

### 1.4.2 Öznitelikler arası ilişki

In [ ]:
from analysis import spearman_corr, plot_correlation
corr = spearman_corr(pool)
show_figure(plot_correlation(corr), "fig_feature_corr", r"Öznitelikler arası Spearman sıra korelasyonu $\rho$.")
show_table(corr, "tab_feature_corr", "Spearman sıra korelasyon matrisi.", floatfmt="{:.2f}")

### 1.4.3 Anlamsal kümeler

In [ ]:
from analysis import cluster_table, plot_cluster_projection
ct = cluster_table(pool)
show_table(ct, "tab_cluster_profile",
           "Küme profilleri: boyut, küme içi ortalama öznitelikler ve küme merkezine en yakın örnek istem.", floatfmt="{:,.2f}")
show_figure(plot_cluster_projection(pool["cluster"].to_numpy()), "fig_cluster_pca",
            "384 boyutlu gömmelerin PCA ile 2B izdüşümü (renk = K-Means kümesi; en çok 8.000 rastgele nokta).")

In [ ]:
import matplotlib.pyplot as plt
sz = pool["cluster"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(sz.index, sz.values, color="#0072B2")
ax.axhline(50, color="#D55E00", ls="--", label="tabakalı seçim kotası (50/küme)")
ax.set_xlabel("anlamsal küme"); ax.set_ylabel("örnek sayısı"); ax.legend()
fig.tight_layout()
show_figure(fig, "fig_cluster_sizes", "K-Means küme boyutları. Her kümede en az 50 örnek bulunması, tabakalı Top-K seçimi için gereklidir.")
assert sz.min() >= 50, "Bir küme 50 örnekten küçük; tabakalı seçim kotası dolmaz."

## 1.5 Tartışma *(deney sonrası doldurulacak)*
Aşağıdaki sorular rapor metninin iskeletini oluşturur; yanıtlar yukarıdaki tablo ve şekillere dayandırılmalıdır.

1. **Uzunluk–olabilirlik ilişkisi:** $L_y$ ile $PPL$ / $H_y$ arasındaki korelasyon işareti ve büyüklüğü nedir? Bu, yalnızca uzunluğa dayalı seçimin bir vekil olup olmadığı konusunda ne söyler?
2. **IFD'nin özgünlüğü:** $IFD$, $PPL$ ile ne kadar örtüşüyor? Düşük $|\rho|$ değeri GP'ye bileşik skor aratmayı gerekçelendirir mi?
3. **Küme yapısı:** Hangi kümeler ortalama olarak daha uzun / daha yüksek IFD'li örnekler içeriyor? Küme merkezine en yakın istemler kümelerin anlamsal temasını (ör. kodlama, özetleme, matematik) doğruluyor mu?
4. **Tehditler:** Öznitelikler 0.5B model üzerinden hesaplandı (3B hedef modele aktarım varsayımı, bkz. defter 03–04); istem/yanıt kırpması uzun örneklerin $L_x,L_y$ değerini etkilemez (ham sayılar saklanır) fakat $PPL, H_y$ yalnızca ilk 512 simgeden hesaplanır.

**Sonraki adım:** `02_gp_evolution` — bu öznitelik matrisi üzerinde GP ile $f_{GP}^*$ aranır.

In [ ]:
# Colab VM'de outputs/ oturum kapanınca silinir: sonuçları yerel makineye indirin.
from common import export_outputs
export_outputs("outputs_01")